<a href="https://colab.research.google.com/github/temesgenaddise/cosc-650-applied-llm-systems/blob/main/Week-7/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week-7 Assignment | The Adaptation Decision and a Production Dataset

*This Notebook Experiments on the healthcare administrative intent classification. Dynamic few-shot selection reuses the Week 5 retrieval pattern: normalized `all-MiniLM-L6-v2` embeddings, inner-product/cosine similarity, and top-k retrieval.*


## Part 1: Prepare the dataset

*Create a domain-specific dataset, remove invalid/duplicate records, make a balanced train/evaluation split, and report basic quality checks before adaptation.*


In [1]:
%pip install -q sentence-transformers faiss-cpu tiktoken openai pandas

import os, re, random, pathlib
from collections import Counter
import pandas as pd

random.seed(7)

INTENTS = {
    'schedule_appointment': (
        'schedule an appointment with {dept} {time}',
        'book me a visit with {dept} {time}',
        'I need an appointment with {dept}',
        'can I see {dept} {time}'
    ),
    'refill_medication': (
        'request a refill for {med}',
        'I need my {med} refilled',
        'send a refill request for {med}',
        'my {med} prescription needs a refill'
    ),
    'get_lab_results': (
        'show my {lab} results',
        'are my {lab} results available',
        'I want to check my {lab} results',
        'where can I see the result of my {lab}'
    ),
    'billing_question': (
        'I have a question about my {bill}',
        'why was I charged for {bill}',
        'help me understand my {bill}',
        'I need billing help with {bill}'
    ),
    'find_clinic': (
        'find a {clinic} clinic near {city}',
        'where is a {clinic} clinic in {city}',
        'locate a {clinic} clinic near me',
        'I need the nearest {clinic} clinic in {city}'
    )
}

SLOT = {
    'dept':['primary care','cardiology','dermatology','pediatrics','orthopedics','neurology'],
    'time':['tomorrow','next week','Monday morning','Friday afternoon','this month'],
    'med':['lisinopril','metformin','atorvastatin','amoxicillin','albuterol','amlodipine','losartan','omeprazole','gabapentin','levothyroxine','sertraline','hydrochlorothiazide','prednisone','azithromycin','insulin'],
    'lab':['blood test','CBC','A1C','cholesterol','urinalysis','CMP','thyroid test','vitamin D','iron panel','kidney function','liver function','COVID test','strep test','lipid panel','glucose test'],
    'bill':['copay','hospital bill','lab charge','clinic visit','insurance balance','deductible','specialist visit','imaging charge','urgent care bill','pharmacy charge','procedure bill','therapy visit','vaccination charge','telehealth visit','emergency visit'],
    'clinic':['primary care','urgent care','pediatric','specialty'],
    'city':['Portland','Beaverton','Gresham','Hillsboro','Tigard']
}

def fill(template):
    return re.sub(r'\{(\w+)\}', lambda m: random.choice(SLOT[m.group(1)]), template)

# Generate enough variation for cleaning, splitting, and retrieval.
raw = [{'text': fill(random.choice(templates)), 'label': label}
       for label, templates in INTENTS.items() for _ in range(180)]

# Add deliberate noise so the quality checks have something real to remove.
raw += [{'text':'', 'label':'schedule_appointment'}] * 4
raw += [{'text':'hello', 'label':'unknown'}] * 3
print('raw pool:', len(raw))


raw pool: 907


In [2]:
VALID = set(INTENTS)
valid = [r for r in raw if r['text'].strip() and r['label'] in VALID]

seen, dedup = set(), []
for r in valid:
    key = (r['text'].lower(), r['label'])
    if key not in seen:
        seen.add(key)
        dedup.append(r)

print(f'dropped invalid {len(raw)-len(valid)}, dropped duplicate {len(valid)-len(dedup)}, clean {len(dedup)}')

by = {label:[r for r in dedup if r['label'] == label] for label in sorted(VALID)}
train, eval_ = [], []
for label, rows in by.items():
    random.shuffle(rows)
    n_train = min(40, max(1, len(rows)-10))
    train += rows[:n_train]
    eval_ += rows[n_train:n_train+10]

random.shuffle(train)
random.shuffle(eval_)
print('train', len(train), 'eval', len(eval_))
print('train label balance:', dict(sorted(Counter(r['label'] for r in train).items())))
print('eval label balance:', dict(sorted(Counter(r['label'] for r in eval_).items())))

import tiktoken
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.')/'.tiktoken_cache').resolve())
enc = tiktoken.get_encoding('cl100k_base')
lens = [len(enc.encode(r['text'])) for r in train]
print(f'token length  min {min(lens)} median {sorted(lens)[len(lens)//2]} max {max(lens)}')
print('format check (text+label present):', all(r['text'] and r['label'] for r in train + eval_))
print('label check:', all(r['label'] in VALID for r in train + eval_))
print('train/eval overlap:', len({r['text'].lower() for r in train} & {r['text'].lower() for r in eval_}))

dataset_df = pd.DataFrame(train + eval_)
display(dataset_df.head())


dropped invalid 7, dropped duplicate 593, clean 307
train 200 eval 50
train label balance: {'billing_question': 40, 'find_clinic': 40, 'get_lab_results': 40, 'refill_medication': 40, 'schedule_appointment': 40}
eval label balance: {'billing_question': 10, 'find_clinic': 10, 'get_lab_results': 10, 'refill_medication': 10, 'schedule_appointment': 10}
token length  min 4 median 8 max 11
format check (text+label present): True
label check: True
train/eval overlap: 0


,text,label
0,find a urgent care clinic near Gresham,find_clinic
1,find a pediatric clinic near Hillsboro,find_clinic
2,send a refill request for insulin,refill_medication
3,request a refill for amlodipine,refill_medication
4,are my glucose test results available,get_lab_results


**Part 1 Result interpretation.**

     The quality report verifies that empty/invalid rows and exact duplicates
     are removed, labels belong to the approved intent set, the train/
     evaluation split is balanced, and there is no text overlap between the two
     splits. Token-length statistics provide a simple prompt-cost check.



## Part 2: Adapt with dynamic few-shot retrieval

*This cell Reuses the Week 5 retrieval design. Training examples are embedded with  all-MiniLM-L6-v2`, normalized, stored in a FAISS inner-product index, and
retrieved by top-k cosine-equivalent similarity for each evaluation input.*


In [3]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

os.environ['HF_HOME'] = str((pathlib.Path('.')/'.hf_cache').resolve())
EMBEDDING_MODEL, TOP_K = 'sentence-transformers/all-MiniLM-L6-v2', 4
embedder = SentenceTransformer(EMBEDDING_MODEL)

train_vectors = embedder.encode(
    [r['text'] for r in train],
    normalize_embeddings=True,
    show_progress_bar=False
).astype('float32')

index = faiss.IndexFlatIP(train_vectors.shape[1])
index.add(train_vectors)

def few_shot(query, k=TOP_K):
    q = embedder.encode([query], normalize_embeddings=True, show_progress_bar=False).astype('float32')
    scores, positions = index.search(q, min(k, len(train)))
    return [{**train[i], 'score': float(score), 'rank': rank + 1}
            for rank, (i, score) in enumerate(zip(positions[0], scores[0])) if i >= 0]

ex = eval_[0]
print('eval input:', ex['text'], '(true:', ex['label'], ')')
print('retrieved few-shot examples:')
for s in few_shot(ex['text']):
    print(f'  rank={s["rank"]} score={s["score"]:.3f} {s["text"]} -> {s["label"]}')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

eval input: where is a specialty clinic in Gresham (true: find_clinic )
retrieved few-shot examples:
  rank=1 score=0.917 where is a primary care clinic in Gresham -> find_clinic
  rank=2 score=0.896 I need the nearest specialty clinic in Gresham -> find_clinic
  rank=3 score=0.882 find a primary care clinic near Gresham -> find_clinic
  rank=4 score=0.849 where is a urgent care clinic in Gresham -> find_clinic


**Part 2 Result interpretation.**

     Because both query and training vectors are L2-normalized, FAISS inner
     product is equivalent to cosine similarity. The retrieved examples are
     therefore the most semantically similar labeled training requests,
     matching the retrieval approach used in Week 5.


## Part 3: Evaluate /Evaluate zero-shot vs. dynamic few-shot, including one failure.

*This cell Runs the same evaluation set in both conditions. Report overall accuracy, per-intent accuracy, and a failure slice.if no intent becomes worse with few-shot prompting, it reports the weakest observed slice.*


In [4]:
%pip install -q -U google-genai scikit-learn

import time
import re
from google import genai
from google.genai import types
from google.colab import userdata
from sklearn.metrics import accuracy_score, f1_score, classification_report

API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("Add GEMINI_API_KEY to Colab Secrets before running the notebook.")

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.8-flash"
print("Native Gemini client ready. Model:", MODEL)

LABELS = sorted(VALID)

def clean_gemini_label(response_text):
    """Normalize Gemini output to one allowed healthcare administrative intent."""
    if response_text is None:
        return "invalid_output"

    text = response_text.strip().lower()
    text = text.replace("`", "").replace("*", "").replace(".", "").strip()

    # Prefer an exact match.
    for label in LABELS:
        if text == label:
            return label

    # Fallback: find one valid label in a slightly longer response.
    for label in sorted(LABELS, key=len, reverse=True):
        if re.search(rf"\b{re.escape(label)}\b", text):
            return label

    return "invalid_output"

def call_gemini(prompt, max_retries=3):
    """Call Gemini with deterministic settings and retry transient failures."""
    for attempt in range(max_retries):
        try:
            safety_settings = [
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
            ]

            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                    max_output_tokens=100,
                    safety_settings=safety_settings,
                    thinking_config=types.ThinkingConfig(thinking_budget=0)
                )
            )

            content = None
            try:
                content = response.text
            except Exception:
                pass

            if not content:
                try:
                    if response.candidates:
                        parts = response.candidates[0].content.parts
                        content = "".join(
                            part.text for part in parts
                            if hasattr(part, "text") and part.text
                        )
                except Exception:
                    pass

            return clean_gemini_label(content)

        except Exception as error:
            if attempt == max_retries - 1:
                print("Gemini call failed:", error)
                return "api_error"
            time.sleep(2 ** attempt)

def build_zero_shot_prompt(text):
    labels = ", ".join(LABELS)
    return f"""You are a healthcare administrative request classifier.

Classify the request into exactly ONE of these intent labels:
{labels}

Intent meanings:
schedule_appointment = book or arrange a healthcare visit
refill_medication = request a prescription medication refill
get_lab_results = obtain or check laboratory test results
billing_question = ask about healthcare charges, bills, balances, or insurance costs
find_clinic = locate a clinic or healthcare facility

Return only the label name and absolutely nothing else.

Text: {text}
Intent:"""

def build_dynamic_prompt(text, k=TOP_K):
    """Use Week_7_2 FAISS retrieval to create the dynamic few-shot prompt."""
    examples = few_shot(text, k=k)
    demo = "\n\n".join(
        f'Example {i+1}\nText: {r["text"]}\nIntent: {r["label"]}'
        for i, r in enumerate(examples)
    )
    labels = ", ".join(LABELS)
    return f"""You are a healthcare administrative request classifier.

Classify the request into exactly ONE of these intent labels:
{labels}

Use the retrieved examples as demonstrations.

{demo}

Now classify:
Text: {text}
Intent:"""

def gemini_zero_shot_predict(text):
    return call_gemini(build_zero_shot_prompt(text))

def gemini_dynamic_predict(text, k=TOP_K):
    return call_gemini(build_dynamic_prompt(text, k=k))

# Small connection test before the full evaluation.
test_prompt = build_zero_shot_prompt(
    "I need an appointment with cardiology next week."
)
print("Gemini connection test decision:", call_gemini(test_prompt))


Native Gemini client ready. Model: gemini-3.8-flash
Gemini connection test decision: schedule_appointment


In [5]:
# Run the same evaluation set under zero-shot and dynamic few-shot conditions.
eval_results = pd.DataFrame(eval_).copy()

print("Generating zero-shot predictions...")
eval_results["zero_shot_pred"] = [
    gemini_zero_shot_predict(text)
    for text in eval_results["text"]
]

print("Generating dynamic few-shot predictions...")
eval_results["dynamic_pred"] = [
    gemini_dynamic_predict(text)
    for text in eval_results["text"]
]

def metric_row(name, pred):
    return {
        "system": name,
        "accuracy": accuracy_score(eval_results["label"], pred),
        "macro_f1": f1_score(
            eval_results["label"],
            pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    }

metrics = pd.DataFrame([
    metric_row("Zero-shot", eval_results["zero_shot_pred"]),
    metric_row(f"Dynamic few-shot (top-{TOP_K})", eval_results["dynamic_pred"])
])
display(metrics)

print("\nZero-shot classification report")
print(classification_report(
    eval_results["label"],
    eval_results["zero_shot_pred"],
    labels=LABELS,
    digits=3,
    zero_division=0
))

print("\nDynamic few-shot classification report")
print(classification_report(
    eval_results["label"],
    eval_results["dynamic_pred"],
    labels=LABELS,
    digits=3,
    zero_division=0
))


Generating zero-shot predictions...
Generating dynamic few-shot predictions...


,system,accuracy,macro_f1
0,Zero-shot,1.00,1.000000
1,Dynamic few-shot (top-4),0.94,0.968421



Zero-shot classification report
                      precision    recall  f1-score   support

    billing_question      1.000     1.000     1.000        10
         find_clinic      1.000     1.000     1.000        10
     get_lab_results      1.000     1.000     1.000        10
   refill_medication      1.000     1.000     1.000        10
schedule_appointment      1.000     1.000     1.000        10

            accuracy                          1.000        50
           macro avg      1.000     1.000     1.000        50
        weighted avg      1.000     1.000     1.000        50


Dynamic few-shot classification report
                      precision    recall  f1-score   support

    billing_question      1.000     0.900     0.947        10
         find_clinic      1.000     1.000     1.000        10
     get_lab_results      1.000     0.900     0.947        10
   refill_medication      1.000     0.900     0.947        10
schedule_appointment      1.000     1.000     1.000    

In [ ]:
# Per-intent slice analysis and visible failure analysis.
slice_results = (
    eval_results.assign(
        zero_correct=eval_results["zero_shot_pred"].eq(eval_results["label"]),
        dynamic_correct=eval_results["dynamic_pred"].eq(eval_results["label"])
    )
    .groupby("label")
    .agg(
        n=("text", "size"),
        zero_shot_accuracy=("zero_correct", "mean"),
        dynamic_few_shot_accuracy=("dynamic_correct", "mean")
    )
    .reset_index()
)

slice_results["difference"] = (
    slice_results["dynamic_few_shot_accuracy"] -
    slice_results["zero_shot_accuracy"]
)

display(slice_results.sort_values("difference"))

failure_slices = slice_results[slice_results["difference"] < 0]

if not failure_slices.empty:
    weakest = failure_slices.sort_values("difference").iloc[0]
    print(
        f'Failure slice: {weakest["label"]} underperformed zero-shot by '
        f'{weakest["difference"]:.1%}.'
    )
else:
    weakest = slice_results.sort_values("difference").iloc[0]
    print(
        "No intent-level slice underperformed zero-shot in this run. "
        f'The weakest observed slice was {weakest["label"]} '
        f'with a change of {weakest["difference"]:+.1%}.'
    )

failure_cases = eval_results[
    eval_results["zero_shot_pred"].eq(eval_results["label"]) &
    ~eval_results["dynamic_pred"].eq(eval_results["label"])
].copy()

print(
    "\nCases where zero-shot was correct but dynamic few-shot failed:",
    len(failure_cases)
)

if len(failure_cases):
    display(
        failure_cases[
            ["text", "label", "zero_shot_pred", "dynamic_pred"]
        ].head(5)
    )
    # Write a review request//////

,label,n,zero_shot_accuracy,dynamic_few_shot_accuracy,difference
0,billing_question,10,1.0,0.9,-0.1
2,get_lab_results,10,1.0,0.9,-0.1
3,refill_medication,10,1.0,0.9,-0.1
1,find_clinic,10,1.0,1.0,0.0
4,schedule_appointment,10,1.0,1.0,0.0


Failure slice: billing_question underperformed zero-shot by -10.0%.

Cases where zero-shot was correct but dynamic few-shot failed: 3


,text,label,zero_shot_pred,dynamic_pred
7,I have a question about my emergency visit,billing_question,billing_question,invalid_output
12,request a refill for levothyroxine,refill_medication,refill_medication,invalid_output
17,where can I see the result of my strep test,get_lab_results,get_lab_results,invalid_output


**Why Dynamic few shot underperformed zero-shot**

     Dynamic few-shot prompting underperformed zero-shot prompting (94% vs. 100%
     accuracy) because the retrieved examples mighty have introduced irrelevant
     or misleading context. Semantic similarity does not always guarantee
     useful examples, potentially causing classification errors. Since the
     zero-shot model already performed well, additional examples provided
     limited benefits. Improving example selection and optimizing top-k
     retrieval could enhance dynamic few-shot performance.


## Part 4: Adaptation decision memo

*Weigh doing nothing, prompt-only adaptation, RAG, dynamic few-shot prompting, and fine-tuning. The memo below is generated from the measured Part 3 results when they are available.*


In [7]:
if eval_results is not None:
    # Extract accuracies from the metrics DataFrame created in Part 3
    zero_acc = metrics.loc[metrics['system'] == 'Zero-shot', 'accuracy'].values[0]
    few_acc = metrics.loc[metrics['system'].str.startswith('Dynamic few-shot'), 'accuracy'].values[0]

    delta = few_acc - zero_acc

    # Use slice_results instead of the undefined slice_df
    weakest_idx = slice_results['difference'].idxmin()
    weakest = slice_results.loc[weakest_idx, 'label']
    weakest_delta = slice_results.loc[weakest_idx, 'difference']

    direction = 'improved' if delta > 0 else ('declined' if delta < 0 else 'did not change')

    memo = f'''Decision memo

Task: classify healthcare administrative requests into five fixed intent labels.

Measured result: zero-shot accuracy was {zero_acc:.1%}; dynamic few-shot accuracy was {few_acc:.1%}. Overall performance {direction} by {delta:+.1%}. The weakest per-intent slice was {weakest}, with a {weakest_delta:+.1%} change.

Doing nothing is the lowest-cost option, but it leaves the baseline at {zero_acc:.1%} and does not use the labeled examples already available. Prompt-only adaptation is inexpensive and useful because the label set can be stated explicitly, but it cannot tailor examples to each incoming request. RAG is valuable when the task requires external factual evidence; this task is classification over a fixed label set, so document retrieval would add complexity without directly solving the main adaptation need. Dynamic few-shot is the best fit for the current experiment because it reuses the Week 5 semantic retrieval method, requires no GPU or model training, and selects examples that are close to each request. Fine-tuning could be reconsidered later if a much larger stable labeled dataset is available and prompt/retrieval performance remains insufficient, but it adds training, validation, deployment, and maintenance cost.

Decision: retain the explicit classification prompt and dynamic few-shot retrieval for the next iteration, while monitoring the weakest slice ({weakest}). Revisit fine-tuning only if repeated evaluation on a larger production dataset shows a persistent accuracy gap that retrieval and prompt changes do not resolve.'''
    print(memo)
else:
    print('Run Part 3 with GEMINI_API_KEY first so this memo can be grounded in measured zero-shot and few-shot results.')

Decision memo

Task: classify healthcare administrative requests into five fixed intent labels.

Measured result: zero-shot accuracy was 100.0%; dynamic few-shot accuracy was 94.0%. Overall performance declined by -6.0%. The weakest per-intent slice was billing_question, with a -10.0% change.

Doing nothing is the lowest-cost option, but it leaves the baseline at 100.0% and does not use the labeled examples already available. Prompt-only adaptation is inexpensive and useful because the label set can be stated explicitly, but it cannot tailor examples to each incoming request. RAG is valuable when the task requires external factual evidence; this task is classification over a fixed label set, so document retrieval would add complexity without directly solving the main adaptation need. Dynamic few-shot is the best fit for the current experiment because it reuses the Week 5 semantic retrieval method, requires no GPU or model training, and selects examples that are close to each request. F

## Submission notes

Commit the completed dataset/notebook, update the agent context file as required by the course, and open the pull request with a short research note summarizing the dataset quality checks, retrieval method, zero-shot vs. dynamic few-shot results, failure slice, and adaptation decision.
